# Benders Decomposition: Robust Basestock Problem Implementation

**Implementation by:** Amin Javanmard

**Original Paper Reference:**

Bienstock, D., & Özbay, N. (2008). Computing robust basestock levels. 
*Discrete Optimization*, 5(2), 389–414. 
https://doi.org/10.1016/j.disopt.2007.10.001

---

## Overview

This notebook implements the Benders decomposition approach from the above reference 
for solving the robust lot-sizing problem with demand uncertainty.

**Key Components:**
- **Master Problem:** Multi-scenario coordination with recourse decisions
- **Adversarial Problem:** Worst-case demand search via MILP
- **Decomposition Framework:** Iterative refinement via cutting planes

## Purpose

This is a **reference implementation** of the published methodology, provided 
for educational purposes and as a foundation for extensions in robust and 
adaptive optimization.

**Note:** All mathematical formulations and algorithmic details are from the 
cited reference. For novel contributions in robust lot-sizing under Markov 
disruptions and more information, contact the author.

---

In [ ]:
#Imports

import numpy as np
import gurobipy as gp
from gurobipy import GRB
import json

In [ ]:
# Function to Load Instances

def load_instance(filename):
    try:
        with open(filename, 'r') as f:
            params = json.load(f)
        # Convert relevant lists back to numpy arrays
        params["mean_normal_demand"] = np.array(params["mean_normal_demand"])
        params["std_normal_demand"] = np.array(params["std_normal_demand"])
        params["mean_disaster_demand"] = np.array(params["mean_disaster_demand"])
        print(f"Instance parameters loaded successfully from {filename}")
        return params
    except FileNotFoundError:
        print(f"ERROR: Instance file not found at {filename}")
        return None
    except Exception as e:
        print(f"ERROR: Failed to load or process instance file {filename}: {e}")
        return None

In [54]:
# Master Problem (MP)

def Master_Problem(all_scenarios, T, p_costs, q_costs, h_costs, C, b_costs):
    print(f" --- Entering Master Problem with {len(all_scenarios)} scenarios ---")
    try:
        model = gp.Model("Master Problem (Multi-Scenario)")
        num_scenarios = len(all_scenarios)
        if num_scenarios == 0:
            print("Error: Master Problem called with zero scenarios.")
            return None, None, None, None # For LB, X, Y, H_mp

        X = model.addVars(T, vtype=GRB.CONTINUOUS, lb=0, name="Production")
        Y = model.addVars(T, vtype=GRB.BINARY, name="Setup")
        H_mp_cost = model.addVars(num_scenarios, T, vtype=GRB.CONTINUOUS, lb=0, name="MP_Holding_Backlog_Cost")
        Theta = model.addVar(vtype=GRB.CONTINUOUS, lb=0, name="Max_Scenario_Cost")

        model.setObjective(Theta, GRB.MINIMIZE)

        for s in range(num_scenarios):
            scenario_prod_setup_cost = gp.quicksum(p_costs[t] * X[t] + q_costs[t] * Y[t] for t in range(T))
            scenario_h_b_cost = gp.quicksum(H_mp_cost[s, t] for t in range(T))
            model.addConstr(Theta >= scenario_prod_setup_cost + scenario_h_b_cost, name=f"Theta_link_s{s}")

    
        for s in range(num_scenarios):
            # For bursty AP, scenario_content[0] is 1D realized demand
            # scenario_content[1] is 1D F_burst (not directly used in MP demand sum)
            realized_demands_s = all_scenarios[s][0]

            # Basic check for 1D demand structure
            if not isinstance(realized_demands_s, (list, np.ndarray)) or \
                (len(realized_demands_s) > 0 and isinstance(realized_demands_s[0], (list, np.ndarray))):
                print(f"ERROR in MP: Scenario {s} has unexpected demand structure. Expected 1D list/array of demands.")
                print(f"Received: {type(realized_demands_s)}")
                return None, None, None, None
            if len(realized_demands_s) != T and T > 0:
                print(f"ERROR in MP: Scenario {s} demand vector length {len(realized_demands_s)} != T {T}")
                return None, None, None, None

            for t in range(T):
                cumulative_production_t = gp.quicksum(X[i] for i in range(t + 1))
                cumulative_demand_s_t = gp.quicksum(realized_demands_s[i] for i in range(t + 1)) if T > 0 else 0
                net_inventory_s_t = cumulative_production_t - cumulative_demand_s_t

                model.addConstr(H_mp_cost[s, t] >= h_costs[t] * net_inventory_s_t, name=f"Holding_s{s}_t{t}")
                model.addConstr(H_mp_cost[s, t] >= -b_costs[t] * net_inventory_s_t, name=f"Backlog_s{s}_t{t}")

        for t in range(T):
            model.addConstr(X[t] <= C * Y[t], name=f"Capacity_{t}")

        model.setParam('OutputFlag', 0) 
        model.optimize()

        if model.status == GRB.OPTIMAL:
            LB = model.objVal
            X_values = [X[t].x for t in range(T)]
            Y_values = [round(Y[t].x) for t in range(T)]
            H_mp_values = [[H_mp_cost[s, t].x for t in range(T)] for s in range(num_scenarios)]
            return LB, X_values, Y_values, H_mp_values
        elif model.status == GRB.INFEASIBLE:
            print("Master Problem is Infeasible.")
            return None, None, None, None
        else:
            print(f"Master Problem Optimization ended with status {model.status}")
            return None, None, None, None
    except gp.GurobiError as e:
        print(f"Gurobi error in Master Problem: Error code {e.errno}: {e}")
        return None, None, None, None
    except Exception as e:
        print(f"An error occurred in Master Problem: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None, None

In [ ]:
# Adversarial Problem (AP) - Bursty Demand Model

def Adversarial_Problem_Bursty(
    X_values, T,
    mean_normal_demand_1D, std_normal_demand_1D,
    peak_demand_1D, Gamma_d_normal, W_window_size,
    h_costs, b_costs
    ):
    print(" --- Entering Adversarial Problem (Bursty Demand - MILP) ---")
    try:
        model = gp.Model("Adversarial_Problem_Bursty")

        # --- Decision Variables ---
        F_burst = model.addVars(T, vtype=GRB.BINARY, name="Is_Burst_Period")
        d_realized = model.addVars(T, vtype=GRB.CONTINUOUS, lb=0, name="Realized_Demand")
        sigma_d_normal_dev = model.addVars(T, vtype=GRB.CONTINUOUS, lb=-1.0, ub=1.0, name="Sigma_Normal_Dev")
        abs_sigma_d_normal_dev = model.addVars(T, vtype=GRB.CONTINUOUS, lb=0, name="Abs_Sigma_Normal_Dev")
        d_normal_potential = model.addVars(T, vtype=GRB.CONTINUOUS, name="Potential_Normal_Demand")
        d_normal_active_component = model.addVars(T, vtype=GRB.CONTINUOUS, lb=0, name="Active_Normal_Demand_Component")

        p_state = model.addVars(T, vtype=GRB.BINARY, name="Inv_State_Indicator_p")
        InvCost = model.addVars(T, vtype=GRB.CONTINUOUS, lb=0, name="HoldingCost_I")
        BacklogCost = model.addVars(T, vtype=GRB.CONTINUOUS, lb=0, name="BacklogCost_B")
        net_inventory = model.addVars(T, vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name="NetInventory")

        model.setObjective(gp.quicksum(InvCost[t] + BacklogCost[t] for t in range(T)), GRB.MAXIMIZE)

        # --- Constraints ---
        
        # 1. Potential Normal Demand and its Uncertainty
        for t in range(T):
            model.addConstr(d_normal_potential[t] == mean_normal_demand_1D[t] + \
                                                  sigma_d_normal_dev[t] * std_normal_demand_1D[t],
                                                    name=f"PotentialNormalDemandDef_{t}")
            model.addConstr(abs_sigma_d_normal_dev[t] >= sigma_d_normal_dev[t], name=f"AbsSigmaD_pos_{t}")
            model.addConstr(abs_sigma_d_normal_dev[t] >= -sigma_d_normal_dev[t], name=f"AbsSigmaD_neg_{t}")
        model.addConstr(gp.quicksum(abs_sigma_d_normal_dev[t] for t in range(T)) <= Gamma_d_normal, "Budget_Normal_Demand_Dev")

        # 2.  Realized Demand d_realized[t] using F_burst[t]
        M_d_norm_upper = {}
        M_d_norm_lower = {}
        for t in range(T):
            M_d_norm_upper[t] = mean_normal_demand_1D[t] + std_normal_demand_1D[t]
            M_d_norm_lower[t] = mean_normal_demand_1D[t] - std_normal_demand_1D[t]
            M_d_norm_lower[t] = max(0, M_d_norm_lower[t]) # Ensure non-negative

        for t in range(T):
            model.addConstr(d_normal_active_component[t] <= M_d_norm_upper[t] * (1 - F_burst[t]), name=f"BigM_d_norm_act_ub1_{t}")
            model.addConstr(d_normal_active_component[t] >= M_d_norm_lower[t] * (1 - F_burst[t]), name=f"BigM_d_norm_act_lb1_{t}")
            model.addConstr(d_normal_active_component[t] <= d_normal_potential[t] - M_d_norm_lower[t] * F_burst[t], name=f"BigM_d_norm_act_ub2_{t}")
            model.addConstr(d_normal_active_component[t] >= d_normal_potential[t] - M_d_norm_upper[t] * F_burst[t], name=f"BigM_d_norm_act_lb2_{t}")
            model.addConstr(d_realized[t] == d_normal_active_component[t] + peak_demand_1D[t] * F_burst[t], name=f"RealizedDemandDef_{t}")

        # 3. Bursty Demand Constraint: At most 1 burst in any W_window_size consecutive periods
        if W_window_size > 0 and W_window_size <= T:
            for k in range(T - W_window_size + 1): # k from 0 to T-W
                model.addConstr(gp.quicksum(F_burst[i] for i in range(k, k + W_window_size)) <= 1, name=f"BurstyWindow_{k}")
        elif W_window_size > T and T > 0 :
            model.addConstr(gp.quicksum(F_burst[t] for t in range(T)) <= 1, name="BurstyWindow_Total")

        # 4. Holding/Backlog Cost Constraints 
        max_cum_prod = sum(X_values) if X_values else 0
        max_poss_realized_d_period = [max(mean_normal_demand_1D[t_calc] + std_normal_demand_1D[t_calc], peak_demand_1D[t_calc]) for t_calc in range(T)] if T > 0 else []
        max_cum_demand_val = sum(max_poss_realized_d_period)
        min_cum_demand_val = sum(max(0, mean_normal_demand_1D[t_calc] - std_normal_demand_1D[t_calc]) for t_calc in range(T)) if T > 0 else 0

        max_abs_inv = max(abs(max_cum_prod - min_cum_demand_val), abs(max_cum_prod - max_cum_demand_val)) if T > 0 else abs(max_cum_prod)
        M_inventory_bound = max_abs_inv * 1.2 + 5000 
        # Ensure h_costs and b_costs are not empty if T=0, though loops won't run
        max_h = max(h_costs) if T > 0 and h_costs else 1 
        max_b = max(b_costs) if T > 0 and b_costs else 1
        M_cost_bound = max(max_h, max_b) * M_inventory_bound * 1.1 + 1000 # Adjusted M_cost_bound
        M_cost_bound = max(M_cost_bound, 1000) # Ensure it's not too small

        for t in range(T):
            cumulative_production_t = sum(X_values[i] for i in range(t + 1))
            cumulative_demand_t = gp.quicksum(d_realized[i] for i in range(t + 1))
            model.addConstr(net_inventory[t] == cumulative_production_t - cumulative_demand_t, name=f"NetInvDef_{t}")

            model.addConstr(InvCost[t] >= h_costs[t] * net_inventory[t], name=f"BigM_I_lb_{t}")
            model.addConstr(InvCost[t] <= h_costs[t] * net_inventory[t]+ h_costs[t]* M_cost_bound * (1-p_state[t]), name=f"BigM_I_ub_{t}")
            model.addConstr(InvCost[t] <= h_costs[t] * p_state[t], name=f"BigM_I_ub2_{t}")
            
            model.addConstr(BacklogCost[t] >= -b_costs[t] * net_inventory[t], name=f"BigM_B_lb_{t}")
            model.addConstr(BacklogCost[t] <= -b_costs[t] * net_inventory[t]+ b_costs[t]*M_cost_bound * p_state[t], name=f"BigM_B_ub_{t}")
            model.addConstr(BacklogCost[t] <=  b_costs[t] * M_cost_bound * (1-p_state[t]), name=f"BigM_B_ub2_{t}")


        # --- Solver Settings ---
        model.setParam(GRB.Param.DualReductions, 0)
        model.setParam(GRB.Param.NumericFocus, 3)
        model.setParam('OutputFlag', 0) 

        # --- Optimize ---
        model.optimize()

        # --- Extract results ---
        obj_val = None; realized_d_values = None; burst_F_indicator_values = None
        if model.status == GRB.OPTIMAL or model.status == GRB.SUBOPTIMAL:
            obj_val = model.objVal
            realized_d_values = [d_realized[t].x for t in range(T)]
            burst_F_indicator_values = [round(F_burst[t].x) for t in range(T)]
            return obj_val, realized_d_values, burst_F_indicator_values
        else:
            print(f"Adversarial problem (Bursty) optimization failed with status {model.status}")
            if model.status == GRB.INFEASIBLE: print("INFEASIBLE.")
            elif model.status == GRB.UNBOUNDED: print("UNBOUNDED.")
            return None, None, None
    except gp.GurobiError as e:
        print(f"Gurobi error in Adversarial_Problem_Bursty: {e.errno}, {e}")
        return None, None, None
    except Exception as e:
        print(f"An error occurred in Adversarial_Problem_Bursty: {e}")
        import traceback; traceback.print_exc()
        return None, None, None

In [ ]:
# Calculate Total Cost

def calculate_total_cost(x_plan, demand_component, F_component_or_burst_indicator, T, p_costs, q_costs, h_costs, b_costs):
    total_cost = 0.0
    if T == 0: return 0.0
    for t in range(T):
        prod_cost_t = p_costs[t] * x_plan[t]
        setup_cost_t = q_costs[t] if x_plan[t] > 1e-6 else 0.0
        cumulative_production_t = sum(x_plan[i] for i in range(t + 1))
        
        if not (isinstance(demand_component, (list, np.ndarray)) and \
                (len(demand_component) == 0 or not isinstance(demand_component[0], (list, np.ndarray)))):
            print(f"Error in calculate_total_cost: Expected 1D demand_component, got {type(demand_component)}")
            return float('inf')
        if len(demand_component) != T:
            print(f"Error in calculate_total_cost: demand_component length mismatch T={T}")
            return float('inf')

        cumulative_demand_t = sum(demand_component[i] for i in range(t + 1))

        net_inventory_t = cumulative_production_t - cumulative_demand_t
        holding_cost = h_costs[t] * max(0, net_inventory_t)
        backlog_cost = b_costs[t] * max(0, -net_inventory_t)
        cost_h_b_t = holding_cost + backlog_cost
        total_cost += prod_cost_t + setup_cost_t + cost_h_b_t
    return total_cost

In [ ]:
# Solution approach:

def run_robust_optimization(params):
    T = params["T"]
    C = params["C"]
    p_cost = params["p"]
    b_cost_param = params["b"]
    h_cost_param = params["h"]
    q_cost = params["q"]
    epsilon = params["epsilon"]

    if T == 0:
        print("T=0, skipping optimization.")
        return {"LB": 0, "GUB": 0, "X": [], "Y": [], "Iterations": 0, "Converged": True, "Demand_Scenarios": [([],[])],
                "X_for_GUB": [], "d_for_GUB": [], "F_for_GUB": []}

    mean_normal_demand_1D = np.array(params["mean_normal_demand"])[:, 1]
    std_normal_demand_1D  = np.array(params["std_normal_demand"])[:, 1]
    peak_demand_1D        = np.array(params["mean_disaster_demand"])[:, 0]
    Gamma_d_normal        = params["Gamma_d"]
    W_window_size         = params.get("W_window_size", 12) # Default if not in JSON

    if "W_window_size" not in params:
        print(f"NOTE: 'W_window_size' not found in instance params, defaulting to {W_window_size}")
    
    global calculate_total_cost, Master_Problem, Adversarial_Problem_Bursty 
    
    print("Starting Robust Optimization (MP) with AP Type: BURSTY DEMAND...")

    initial_realized_demands_1D = [mean_normal_demand_1D[t_idx] for t_idx in range(T)]
    initial_F_burst_1D = [0 for _ in range(T)] # No bursts initially
    demand_scenarios = [(initial_realized_demands_1D, initial_F_burst_1D)]

    GUB = float('inf'); LB = -float('inf')
    X_values = None; Y_values = None; H_mp_values = None
    X_for_GUB = None; d_for_GUB = None; F_for_GUB = None

    iteration = 0
    max_iterations = params.get("max_iterations", 200)
    print(f"Initial Scenario Count: {len(demand_scenarios)}")
    gap = float('inf')

    while True:
        print(f"\n--- Iteration {iteration} ---")
        LB_new, X_values_new, Y_values_new, H_mp_values_new = Master_Problem(
            demand_scenarios, T, p_cost, q_cost, h_cost_param, C, b_cost_param 
        )
        if LB_new is None or X_values_new is None or Y_values_new is None:
            print(f"Iteration {iteration}: Master_Problem failed. Stopping.")
            X_values = None; break
        LB = LB_new; X_values = X_values_new; Y_values = Y_values_new; H_mp_values = H_mp_values_new
        print(f"Master Problem Solved. Current LB (Theta) = {LB:.2f}")

        UB_adv, worst_d_realized, worst_F_burst = Adversarial_Problem_Bursty(
            X_values, T,
            mean_normal_demand_1D, std_normal_demand_1D,
            peak_demand_1D, Gamma_d_normal, W_window_size,
            h_cost_param, b_cost_param
        )
        if UB_adv is None or worst_d_realized is None or worst_F_burst is None:
            print(f"Iteration {iteration}: Adversarial_Problem_Bursty failed. Stopping.")
            break
        else:
            print(f"Adversarial Problem (Bursty) Solved. Max Cost (AP Obj) = {UB_adv:.2f}")
            print(f"  Worst-Case Burst Pattern (1=Burst): {worst_F_burst}")

        current_total_cost = calculate_total_cost(
            X_values, worst_d_realized, worst_F_burst, T,
            p_cost, q_cost, h_cost_param, b_cost_param 
        )
        if current_total_cost < GUB:
            GUB = current_total_cost
            X_for_GUB = X_values[:]
            d_for_GUB = worst_d_realized
            F_for_GUB = worst_F_burst
        print(f"Total Cost for current X under its worst case = {current_total_cost:.2f}")
        print(f"Updated GUB = {GUB:.2f}")

        demand_scenarios.append((worst_d_realized, worst_F_burst))

        if LB > 1e-9:
            if GUB != float('inf'): gap = (GUB - LB) / LB
            print(f"Iteration {iteration}: Current Gap = {gap:.2%}" if GUB != float('inf') else f"Iteration {iteration}: GUB is inf, LB={LB:.2f}")
            if 0 <= gap <= epsilon:
                print(f"\nConvergence threshold ({epsilon:.2%}) reached at iteration {iteration}.")
                break
            if GUB < LB - 1e-6 : print(f"WARNING: GUB ({GUB:.4f}) < LB ({LB:.4f}). Check calculations.")
        elif GUB != float('inf'):
            print(f"LB ({LB:.2f}) near zero/negative, using absolute diff. GUB={GUB:.2f}")
            if abs(GUB - LB) <= epsilon:
                print(f"\nConvergence threshold (absolute diff <= {epsilon}) reached at iteration {iteration}.")
                break
        else: print(f"Iteration {iteration}: LB near zero/negative, GUB is inf. Gap not meaningful.")
        
        iteration += 1
        if iteration >= max_iterations:
            print(f"\nReached maximum iterations ({max_iterations}). Stopping.")
            break
            
    converged_status = False
    if GUB != float('inf') and LB != float('-inf'):
        if LB > 1e-9: 
            if GUB != float('inf'): # Ensure GUB is not inf before division
                final_gap_check = (GUB - LB) / LB
                converged_status = (0 <= final_gap_check <= epsilon)
            else: # GUB is inf, LB is positive -> not converged
                converged_status = False
        else: # LB is near zero or negative
            if GUB != float('inf'): # GUB is finite
                converged_status = (abs(GUB - LB) <= epsilon)
            else: # Both are problematic
                converged_status = False 
    
    if X_values is not None:
        results = {
            "LB": LB, "GUB": GUB, "X": X_values, "Y": Y_values, "H_final_MP": H_mp_values,
            "Iterations": iteration, "Converged": converged_status,
            "Demand_Scenarios": demand_scenarios,
            "X_for_GUB": X_for_GUB, "d_for_GUB": d_for_GUB, "F_for_GUB": F_for_GUB
        }
        return results
    else:
        print("Robust Optimization failed to find a solution.")
        return None

In [ ]:
# Main Execution

if __name__ == "__main__":

    # Instance file should be in same directory or data/ folder
    instance_filename_to_run = "data/base_case_T24.json"
    # Or if users run from tutorials/ folder:
    # instance_filename_to_run = "../data/base_case_T24.json"

    print("To generate custom instances, use the instance_generator notebook")
    
    print(f"Loading instance from: {instance_filename_to_run}")
    instance_params = load_instance(instance_filename_to_run)

    if instance_params:
        if "W_window_size" not in instance_params: 
            instance_params["W_window_size"] = 12 
            print(f"NOTE: 'W_window_size' not found in instance params, defaulting to {instance_params['W_window_size']}")

        print(f"\nRunning optimization for T={instance_params['T']} using BURSTY DEMAND ADVERSARY...")
        final_results = run_robust_optimization(instance_params) 

        if final_results:
            print("\n--------------------------------------------------")
            print(f"Robust Optimization Finished (Bursty Demand AP) for Instance: {instance_filename_to_run}")
            print(f"Parameters Used (subset): Gamma_d_normal={instance_params['Gamma_d']:.2f}, W_window={instance_params['W_window_size']}")
            print("--------------------------------------------------")
            print(f"Convergence Status: {'Converged' if final_results['Converged'] else 'Not Converged'}")
            print(f"Iterations Ran: {final_results['Iterations']}")
            print(f"Final GUB: {final_results['GUB']:.2f}")
            print(f"Final LB: {final_results['LB']:.2f}")
            print("\nFinal Production Plan (X_values):")
            print(f"  {[round(x, 2) for x in final_results['X']]}")
            print("\nFinal Setup Decisions (Y_values):")
            print(f"  {final_results['Y']}")
            print("--------------------------------------------------")
            print("\nWorst-Case Scenarios Considered by MP (Bursty Demand Model):")
            print("(Scenario shows (Realized Demands, Burst Pattern (1=Burst)))")
            all_scenarios_data = final_results.get("Demand_Scenarios", [])
            T_actual = instance_params.get("T",0)
            if all_scenarios_data and T_actual > 0:
                for i, (d_scen, F_scen_burst) in enumerate(all_scenarios_data):
                    iter_label = "Initial" if i == 0 else f"Iter{i-1} WC"
                    d_print = [round(val,1) for val in d_scen] if isinstance(d_scen, (list, np.ndarray)) else str(d_scen)
                    print(f"  {iter_label:<8}: D={d_print}, F_burst={F_scen_burst}")
            else: print("  No scenario data to display.")
            print("\nDetails of Scenario that Defined Final GUB (Bursty Model):")
            if final_results.get("d_for_GUB") and final_results.get("F_for_GUB"):
                d_gub_print = [round(d_val,1) for d_val in final_results['d_for_GUB']] if isinstance(final_results['d_for_GUB'], (list, np.ndarray)) else str(final_results['d_for_GUB'])
                print(f"  Realized Demands (d_for_GUB): {d_gub_print}")
                print(f"  Burst Pattern (F_for_GUB):    {final_results['F_for_GUB']}")
                if final_results.get("X_for_GUB"):
                    print(f"  Plan for this GUB (X_for_GUB): {[round(x,1) for x in final_results['X_for_GUB']]}")
            print("--------------------------------------------------")
        else: print("\nOptimization run failed.")
    else: print("Could not run optimization due to instance loading error.")

Loading instance from: instances/base_case_T24.json
Instance parameters loaded successfully from instances/base_case_T24.json
NOTE: 'W_window_size' not found in instance params, defaulting to 12

Running optimization for T=24 using BURSTY DEMAND ADVERSARY...
Starting Robust Optimization (MP) with AP Type: BURSTY DEMAND...
Initial Scenario Count: 1

--- Iteration 0 ---
 --- Entering Master Problem with 1 scenarios ---
Master Problem Solved. Current LB (Theta) = 43464.00
 --- Entering Adversarial Problem (Bursty Demand - MILP) ---
Set parameter DualReductions to value 0
Set parameter NumericFocus to value 3
Set parameter NumericFocus to value 3
Adversarial Problem (Bursty) Solved. Max Cost (AP Obj) = 646800.00
  Worst-Case Burst Pattern (1=Burst): [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Total Cost for current X under its worst case = 690264.00
Updated GUB = 690264.00
Iteration 0: Current Gap = 1488.13%

--- Iteration 1 ---
 --- Entering Master Problem wit

In [ ]:
# The robust solution's cost (This model)

print(f"Total cost of robust solution: {final_results['GUB']:.2f}")

# Final Worst-Case Demand Pattern
print("\nDemand realization in final (worst-case) scenario:")
print(f"  Realized Demands (d_for_GUB): {d_gub_print}")


Total cost of robust solution: 107021.00

Demand realization in final (worst-case) scenario:
  Realized Demands (d_for_GUB): [300.0, 99.0, 100.0, 100.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 81.0, 300.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0]


In [59]:
# Cost in All normal scenario

# Calculate total cost with normal demand in all periods
normal_demand = [instance_params['mean_normal_demand'][t][1] for t in range(T_actual)]  # 80 for all periods
normal_burst_indicator = [0] * T_actual  # No bursts for normal demand

normal_total_cost = calculate_total_cost(
    final_results['X'],           # Production plan from solution
    normal_demand,                # Normal demand (80 in all periods)
    normal_burst_indicator,       # No bursts (all zeros)
    T_actual,                     # Number of periods (12)
    instance_params['p'],         # Production costs
    instance_params['q'],         # Setup costs
    instance_params['h'],         # Holding costs
    instance_params['b']          # Backlog costs
)

print(f"Total cost with normal demand in all periods: {normal_total_cost:.2f}")

# Print demand realization
print(f"\nNormal demand scenario:")
print(f"Demand values:     {[round(d, 2) for d in normal_demand]}")


Total cost with normal demand in all periods: 106951.00

Normal demand scenario:
Demand values:     [80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0]


In [ ]:
# Cost in All disaster scenario

# Calculate total cost with bursty demand in all periods
bursty_demand = [instance_params['mean_disaster_demand'][t][0] for t in range(T_actual)]  # 300 for all periods
bursty_burst_indicator = [1] * T_actual  # All periods are bursts

bursty_total_cost = calculate_total_cost(
    final_results['X'],           # Production plan from solution
    bursty_demand,                # Bursty demand (300 in all periods)
    bursty_burst_indicator,       # All bursts (all ones)
    T_actual,                     # Number of periods (12)
    instance_params['p'],         # Production costs
    instance_params['q'],         # Setup costs
    instance_params['h'],         # Holding costs
    instance_params['b']          # Backlog costs
)

print(f"Total cost with bursty demand in all periods: {bursty_total_cost:.2f}")

# Print demand realization
print(f"Demand values:     {[round(d, 2) for d in bursty_demand]}")


Total cost with bursty demand in all periods: 4080151.00
Demand values:     [300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0, 300.0]


In [ ]:
# Cost with custom demand scenario
custom_demand = normal_demand.copy()  # Start with normal demand
custom_demand[0] = 100  # Set first period demand to 100
custom_demand[-1] = instance_params['mean_disaster_demand'][-1][0]  # Set last period to bursty demand

custom_burst_indicator = [0] * T_actual  # Start with no bursts
custom_burst_indicator[-1] = 1  # Set last period to burst

custom_total_cost = calculate_total_cost(
    final_results['X'],           # Production plan from solution
    custom_demand,                # Custom demand scenario
    custom_burst_indicator,       # Only last period is burst
    T_actual,                     # Number of periods (12)
    instance_params['p'],         # Production costs
    instance_params['q'],         # Setup costs
    instance_params['h'],         # Holding costs
    instance_params['b']          # Backlog costs
)

print(f"Total cost with custom scenario: {custom_total_cost:.2f}")

# Print demand realization
print("\nCustom scenario details:")
print(f"Demand values:     {[round(d, 2) for d in custom_demand]}")




Total cost with custom scenario: 103591.00

Custom scenario details:
Demand values:     [100, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 300.0]


In [63]:
# Compare all scenarios' cost

print("\nComparison of different scenarios:")
print(f"Normal demand only:             {normal_total_cost:.2f}")
print(f"One burst at first period:      {custom_total_cost:.2f}")
print(f"All periods are bursts:         {bursty_total_cost:.2f}")
print(f"Robust solution worst case:     {final_results['GUB']:.2f}")


Comparison of different scenarios:
Normal demand only:             106951.00
One burst at first period:      103591.00
All periods are bursts:         4080151.00
Robust solution worst case:     107021.00
